# Step 2: Filter notebook

In [31]:
import geopandas as gpd
import pandas as pd
from functools import reduce
import osmnx as ox
from shapely.geometry import Point
import rasterio
import numpy as np
import os
import folium
import json
from folium import Choropleth, CircleMarker, GeoJson
import branca.colormap as cm
from IPython.display import display
pd.set_option('display.max_columns', None)

from feature_to_network import *

## Imports

#### Import des segments

In [32]:
operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

print("Set parameters : GG or GE, bike or walk")
territory = 'GG' # GG or GE
network = "bike"  # walk or bike

if territory == 'GG':
    input_file_path = '../../Data/input'
    output_step1_path='../../Data/output/GG/step-1'
    output_step2_path='../../Data/output/GG/step-2'
    output_step3_path='../../Data/output/GG/step-3'


    save_path = '../../Data/output/GG/step-2'

    attribute_source_path = 'source_path_GG'
    file_name = 'file_name_GG'  # column name in attributs_info excel

if territory == 'GE':
    input_file_path = '../../Data/input'
    output_step1_path='../../Data/output/GE/step-1'
    output_step2_path='../../Data/output/GE/step-2'
    output_step3_path='../../Data/output/GE/step-3'


    save_path = '../../Data/output/GE/step-2'

    attribute_source_path = 'source_path_GE'
    file_name = 'file_name_GE'  # column name in attributs_info excel

save_filtered_attributes = True


# Load segments GeoDataFrame (with 'segment_id')
print("Loading segments...")
# reLoad pedestrian segments
segmented_net = gpd.read_parquet(os.path.join(output_step1_path, "step1_all_segments.parquet"))
segmented_net["geometry"] = segmented_net["geometry"].apply(lambda geom: geom.buffer(0) if not geom.is_valid else geom)
segmented_net = segmented_net.to_crs(operation_crs)


Set parameters : GG or GE, bike or walk
Loading segments...


In [33]:
import geopandas as gpd
from shapely.geometry.base import BaseGeometry

# Clean geometries safely
def clean_any_geom(geom):
    if geom is None:
        return None
    if not isinstance(geom, BaseGeometry):
        return None
    if geom.is_valid:
        return geom
    try:
        fixed = geom.buffer(0)
        if fixed is None or fixed.is_empty:
            return geom
        return fixed
    except Exception:
        return geom
        




# Define a function to save the filtered data
def save(save_filtered_attributes, row, gdf, attribute):

    if not save_filtered_attributes:
        print("Note : Save option is disabled.")
        return

    # Parse save formats: comma-separated list allowed
    raw = str(row.get('save_format', '') or '')
    formats = [f.strip().lower() for f in raw.split(',') if f.strip()]
    if not formats:
        formats = ['csv']  # default fallback

    for fmt in formats:
        try:
            if fmt == 'parquet':
                dirpath = f'{output_step2_path}/parquet_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_parquet(f"{dirpath}/{attribute}.parquet")
                print(f"Filtered data saved for attribute: {attribute} in format: parquet")

            elif fmt == 'gpkg' or fmt == 'geopackage':
                dirpath = f'{output_step2_path}/gpkg_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_file(f"{dirpath}/{attribute}.gpkg", driver="GPKG")
                print(f"Filtered data saved for attribute: {attribute} in format: gpkg")

            elif fmt == 'csv':
                dirpath = f'{output_step2_path}/csv_attributs'
                os.makedirs(dirpath, exist_ok=True)
                # to_csv may not preserve geometry consistently; keep original behavior
                gdf.to_csv(f"{dirpath}/{attribute}.csv", index=False)
                print(f"Filtered data saved for attribute: {attribute} in format: csv")

            else:
                # Unknown format -> fallback to csv and warn
                dirpath = f'{output_step2_path}/csv_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_csv(f"{dirpath}/{attribute}.csv", index=False)
                print(f"Warning: Unknown save format '{fmt}' for attribute {attribute}. Data saved as csv.")
        except Exception as e:
            print(f"Error saving {attribute} as {fmt}: {e}")


#### Import des attributs 

In [34]:
attributs_info = pd.read_excel(f"{input_file_path}/attributs/attributs_info.xlsx", sheet_name="attributs_info")
attributs_info = attributs_info[attributs_info['include_in_index'] != False]

In [35]:
attributs_info

,Class,meta_attribute,attribute,include_in_index,source_type_GE,source_path_GE,file_name_GE,source_type_GG,source_path_GG,file_name_GG,merge_rule_GG,initial_weight,class_weight,buffer_size,impact_attribut,geometry_type,how,value_column,clip,filter_column,filter_values,crs,save_format,Unnamed: 23,commentaire
0,comfort,temperature,temperature,True,sitg,attributs/GE/temperature,CLIMAT_TEMPERATURE_14H00_P1_2020/CLIMAT_TEMPER...,opendata,attributs/GG/temperature,20250629_102250.LST.tif,prefer_opendata,0.5,0.5,1.0,defavorable,point,raster,temperature,NaN,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN,NaN
1,attractivite,stationnement_velo,stationnement_velo,True,sitg,NaN,NaN,gpkg_layer,attributs/GG/osm,osm_attributes.gpkg,prefer_opendata,0.5,0.5,10.0,favorable,point,count,NaN,NaN,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN,NaN
2,attractivite,borne_reparation,borne_reparation,True,sitg,NaN,NaN,gpkg_layer,attributs/GG/osm,osm_attributes.gpkg,prefer_opendata,0.5,0.5,10.0,favorable,point,count,NaN,NaN,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN,NaN
4,attractivite,amenites,amenites,True,sitg,NaN,NaN,gpkg_layer,attributs/GG/osm,osm_attributes.gpkg,prefer_opendata,0.5,0.5,10.0,favorable,point,count,NaN,10.0,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN,NaN
7,infrastructure,connectivite,connectivite,True,NaN,NaN,NaN,NaN,NaN,NaN,prefer_opendata,0.5,0.5,10.0,favorable,line,sum,conn_branching_in_buffer,NaN,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN,NaN
9,infrastructure,revetement,revetement,True,NaN,NaN,NaN,gpkg_layer,attributs/GG/osm,osm_attributes.gpkg,prefer_opendata,0.5,0.5,10.0,favorable,line,sum,surface_score,NaN,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN,NaN
11,securite,eclairage,eclairage,True,NaN,NaN,NaN,gpkg_layer,attributs/GG/osm,osm_attributes.gpkg,prefer_opendata,0.5,0.5,10.0,favorable,line,length_ratio,NaN,NaN,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN,NaN
12,securite,vitesse,zone_apaisee,True,sitg,NaN,NaN,gpkg_layer,attributs/GG/osm,osm_attributes.gpkg,prefer_opendata,0.5,0.5,10.0,favorable,line,length_ratio,NaN,NaN,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN,NaN
16,securite,vitesse,vitesse_motorisee,True,NaN,NaN,NaN,gpkg_layer,attributs/GG/osm,osm_attributes.gpkg,prefer_opendata,0.5,0.5,10.0,defavorable,line,length_ratio,NaN,NaN,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN,NaN


In [36]:
import fiona

gpkg_path = f"{input_file_path}/attributs/GG/osm/osm_attributes.gpkg"
print(fiona.listlayers(gpkg_path))


['borne_reparation', 'stationnement_velo', 'location', 'amenites', 'crossing', 'traffic_signals', 'barrier', 'traffic_calming', 'piste', 'bande', 'oneway', 'revetement', 'etat_chaussee', 'eclairage', 'largeur', 'pente', 'conflicts_md', 'vitesse']


**Attributs OSM**

**Attribut stationnement velo**


Amélioration: donner un meilleur score selon la capacité, présence abris ou non

In [38]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'stationnement_velo'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: stationnement_velo
Filters applied
Proportion of features stationnement_velo kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Filtered data saved for attribute: stationnement_velo in format: parquet
Filtered data saved for attribute: stationnement_velo in format: csv
Filtered data saved for attribute: stationnement_velo in format: gpkg


**Attribut zone apaisée**

In [39]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'zone_apaisee'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.maxspeed=='10', 'filtered'] = 1
gdf.loc[gdf.maxspeed=='20', 'filtered'] = 1
gdf.loc[gdf.maxspeed=='30', 'filtered'] = 1
gdf.loc[gdf.maxspeed=='FR:urban', 'filtered'] = 1
gdf.loc[gdf.maxspeed=='CH:urban', 'filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: zone_apaisee
Filters applied
Proportion of features zone_apaisee kept after filtering:
filtered
0    0.600414
1    0.399586
Name: proportion, dtype: float64
Filtered data saved for attribute: zone_apaisee in format: parquet
Filtered data saved for attribute: zone_apaisee in format: csv
Filtered data saved for attribute: zone_apaisee in format: gpkg


**Atttribut Vitesse motorisée**

In [40]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'vitesse_motorisee'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.maxspeed=='40', 'filtered'] = 1
gdf.loc[gdf.maxspeed=='50', 'filtered'] = 1
gdf.loc[gdf.maxspeed=='60', 'filtered'] = 1
gdf.loc[gdf.maxspeed=='70', 'filtered'] = 1
gdf.loc[gdf.maxspeed=='80', 'filtered'] = 1
gdf.loc[gdf.maxspeed=='90', 'filtered'] = 1
gdf.loc[gdf.maxspeed=='FR:rural', 'filtered'] = 1
gdf.loc[gdf.maxspeed=='CH:rural', 'filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: vitesse_motorisee
Filters applied
Proportion of features vitesse_motorisee kept after filtering:
filtered
1    0.598011
0    0.401989
Name: proportion, dtype: float64
Filtered data saved for attribute: vitesse_motorisee in format: parquet
Filtered data saved for attribute: vitesse_motorisee in format: csv
Filtered data saved for attribute: vitesse_motorisee in format: gpkg


**Attribut revêtement**

In [41]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'revetement'


###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
surface_score = {
    "asphalt": 1,
    "concrete": 1,
    "concrete:plates": 1,
    "concrete:lanes": 1,
    "paved": 0,
    "paving_stones": 0,
    "sett": 0,
    "metal": 0,
    "wood": 0,

    "fine_gravel": 0,
    "compacted": 0,
    "gravel": 0,
    "pebblestone": 0,

    "ground": 0,
    "dirt": 0,
    "earth": 0,
    "grass": 0,
    "grass_paver": 0,
    "sand": 0,
    "rock": 0,
    "unpaved": 0,
    "woodchips": 0,
}


gdf["surface_score"] = gdf["surface"].map(surface_score)
gdf.loc[gdf["surface_score"] == 1, "filtered"] = 1


###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: revetement
Filters applied
Proportion of features revetement kept after filtering:
filtered
1    0.818184
0    0.181816
Name: proportion, dtype: float64
Filtered data saved for attribute: revetement in format: parquet
Filtered data saved for attribute: revetement in format: csv
Filtered data saved for attribute: revetement in format: gpkg


**Attribut Aménité**

In [42]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'amenites'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
if "name" in gdf.columns:
    gdf = gdf.drop(columns=["name"])

print(gdf.columns)

gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Index(['osmid', 'amenity', 'shop', 'geometry'], dtype='object')
Processing attribute: amenites
Filters applied
Proportion of features amenites kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Filtered data saved for attribute: amenites in format: parquet
Filtered data saved for attribute: amenites in format: csv
Filtered data saved for attribute: amenites in format: gpkg


**Attribut borne réparation**

In [43]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'borne_reparation'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf['filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: borne_reparation
Filters applied
Proportion of features borne_reparation kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Filtered data saved for attribute: borne_reparation in format: parquet
Filtered data saved for attribute: borne_reparation in format: csv
Filtered data saved for attribute: borne_reparation in format: gpkg


**Attribut éclairage**

In [44]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'eclairage'

###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------
gdf.loc[gdf.lit=='yes', 'filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: eclairage
Filters applied
Proportion of features eclairage kept after filtering:
filtered
1    0.847843
0    0.152157
Name: proportion, dtype: float64
Filtered data saved for attribute: eclairage in format: parquet
Filtered data saved for attribute: eclairage in format: csv
Filtered data saved for attribute: eclairage in format: gpkg


**Attribut Confort thermique**

In [45]:
# Initialize

attribute = 'temperature'
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
raster_path = (f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")

with rasterio.open(raster_path) as src:
    # Read the raster data
    raster_data = src.read(1)  # Read first band
    
    # Define NoData
    nodata_val = src.nodata if src.nodata is not None else -9999
    
    # Valid mask: remove NoData
    valid_mask = (raster_data != nodata_val)
    
    # Get coordinates of valid points
    rows, cols = np.where(valid_mask)
    
    # Sample every 10th point
    rows, cols = rows[::10], cols[::10]
    
    # Get coordinates in map units
    xs, ys = rasterio.transform.xy(src.transform, rows, cols)
    temp_values = raster_data[rows, cols]
    
    # Create GeoDataFrame directly with filtered points
    gdf = gpd.GeoDataFrame({
        'temperature': temp_values,
        'filtered': 1,  # All points are filtered since we pre-filtered the data
        'geometry': [Point(x, y) for x, y in zip(xs, ys)]
    }, geometry='geometry')
    
    # Set CRS and convert to target CRS
    gdf = gdf.set_crs(src.crs)
    gdf = gdf.to_crs(target_crs)

print(f"Processing attribute: {attribute}")
print("\nDescriptive statistics:")
print(gdf['temperature'].describe())

# Save
save(save_filtered_attributes, row, gdf, attribute)

Processing attribute: temperature

Descriptive statistics:
count    406794.000000
mean         31.492273
std          10.037497
min           0.000000
25%          30.278126
50%          33.369716
75%          36.936945
max          54.971017
Name: temperature, dtype: float64
Filtered data saved for attribute: temperature in format: parquet
Filtered data saved for attribute: temperature in format: csv
Filtered data saved for attribute: temperature in format: gpkg
